Task1 FIAT

The main problem is trying to calculate how many cars each models should be produced so that it can get the maximun profit. Therefore, we need to figure out the net profit that each model will get.

In [8]:
from stat import FILE_ATTRIBUTE_ARCHIVE

models=["Panda", "500", "Musa", "Giulia"]
price={"Panda":106, "500":136, "Musa":150, "Giulia":427}
tax={"Panda":0.30, "500":0.15, "Musa":0.20, "Giulia":0.30}
material= {"Panda": 0.57, "500": 0.60, "Musa": 0.55, "Giulia": 0.45}
salary={"Panda": 20000, "500": 11000, "Musa": 20000, "Giulia": 26000}
manhour={"Panda": 40, "500": 45, "Musa": 38, "Giulia": 100}
min_prod={"Panda": 120000, "500": 100000, "Musa": 80000, "Giulia": 15000}
hour_per_month=160
c={} #net income
cost={} # cost
for m in models:
    material_cost=price[m]*material[m]
    labor_cost_per_hour=salary[m]/hour_per_month/1000
    labor_cost=labor_cost_per_hour*manhour[m]
    pretax_profit=price[m]-material_cost-labor_cost
    net_profit=pretax_profit*(1-tax[m])
    c[m] = net_profit
    cost[m]=labor_cost+material_cost
    print(f"{m}: material={material_cost:.2f}, labor={labor_cost:.2f}, "
          f"pretax={pretax_profit:.2f}, net_profit={net_profit:.2f}")

Panda: material=60.42, labor=5.00, pretax=40.58, net_profit=28.41
500: material=81.60, labor=3.09, pretax=51.31, net_profit=43.61
Musa: material=82.50, labor=4.75, pretax=62.75, net_profit=50.20
Giulia: material=192.15, labor=16.25, pretax=218.60, net_profit=153.02


The objective function is

max Z=c1x1+c2x2+c3x3+c4x4

with some cost constraint


In [9]:
import gurobipy as gp
from gurobipy import GRB
m=gp.Model("FIAT")
x=m.addVars(models,name='x')
m.setObjective(gp.quicksum(c[i] * x[i] for i in models), GRB.MAXIMIZE)
m.addConstr(gp.quicksum(cost[i] * x[i] for i in models) <= 4e7, "budget")
for i in models:
    m.addConstr(x[i] >= min_prod[i], f"minprod_{i}")
m.addConstr(x["Panda"] + x["Musa"] <= 300000, "marketing")
m.optimize()
for i in models:
    print(i, x[i].X)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 6 rows, 4 columns and 10 nonzeros
Model fingerprint: 0xa94ff6bc
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [3e+01, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 4e+07]
Presolve removed 6 rows and 4 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.4048076e+07   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.404807577e+07
Panda 120000.0
500 100000.0
Musa 80000.0
Giulia 80135.43666026872


In [10]:
# 查看约束的右端项(RHS)允许变化的范围，不影响最优基/解结构
for c in m.getConstrs():
    print(c.ConstrName, "RHS =", c.RHS,
          "range:[", c.SARHSLow, ",", c.SARHSUp, "]")

# 查看目标函数系数(即c_i)允许变化的范围
for v in m.getVars():
    print(v.VarName, "coef =", v.Obj,
          "range:[", v.SAObjLow, ",", v.SAObjUp, "]")

budget RHS = 40000000.0 range:[ 26425775.0 , inf ]
minprod_Panda RHS = 120000.0 range:[ -0.0 , 220000.0 ]
minprod_500 RHS = 100000.0 range:[ -0.0 , 260274.2233045532 ]
minprod_Musa RHS = 80000.0 range:[ -0.0 , 180000.0 ]
minprod_Giulia RHS = 15000.0 range:[ -inf , 80135.43666026872 ]
marketing RHS = 300000.0 range:[ 200000.0 , inf ]
x[Panda] coef = 28.406000000000002 range:[ -inf , 48.035357005758144 ]
x[500] coef = 43.610312500000006 range:[ -inf , 62.18732065738963 ]
x[Musa] coef = 50.2 range:[ -inf , 64.0642754318618 ]
x[Giulia] coef = 153.01999999999998 range:[ 119.9046418338109 , inf ]


Task2 The Global Market

We need to recgonize the decision variables, x_ij, i represents the model, j represents the country that the cars will be sold. Therefore, i belongs to the set { Panda, 500, Musa, Guilia}, j belongs to the set {Sweden, Italy, Poland, US}. X_ij is the number that each model should be sold in different country.
Thus, we need to calculate the net profit that per unit model sold in the 4 countries.

In [11]:
models = ["Panda", "500", "Musa", "Giulia"]
countries = ["Poland", "Italy", "US", "Sweden"]
data = {
    ("Panda","Poland"):  (0.8,  86,  75),
    ("Panda","Italy"):   (0,    106, 35),
    ("Panda","US"):      (3.5, 150, 40),
    ("Panda","Sweden"):  (2, 112, 2),

    ("500","Poland"):    (0,    92,  20),
    ("500","Italy"):     (1, 136, 40),
    ("500","US"):        (2.8, 170, 50),
    ("500","Sweden"):    (1.6, 150, 5),

    ("Musa","Poland"):   (12,   100, 10),
    ("Musa","Italy"):    (0,    150, 80),
    ("Musa","Sweden"):   (2.2, 170, 1),

    ("Giulia","Italy"):  (0,    427, 8),
    ("Giulia","US"):     (5, 550, 3),
    ("Giulia","Sweden"): (2.5, 500, 1),
}
c_market = {}
demand = {}
total_cost_market = {}
for (i,j),(dl,mkt_price,dm) in data.items():
    if j=="US":
       export_tax=0.025*mkt_price
    else:
        export_tax=0
    total_cost_market[(i, j)] = cost[i] + dl + export_tax
    pretax_profit=mkt_price-total_cost_market[(i, j)]
    net_profit=pretax_profit*(1-tax[i])
    c_market[(i, j)] = net_profit
    demand[(i, j)] = dm
    print(f"{i:8s} -> {j:8s}: cost(material+labor)={cost[i]:.3f} kkr, "
          f"delivery={dl:.3f} kkr, export_tax={export_tax:.3f} kkr, "
          f"pretax={pretax_profit:.3f} kkr, net_profit={net_profit:.3f} kkr, demand={dm}")




Panda    -> Poland  : cost(material+labor)=65.420 kkr, delivery=0.800 kkr, export_tax=0.000 kkr, pretax=19.780 kkr, net_profit=13.846 kkr, demand=75
Panda    -> Italy   : cost(material+labor)=65.420 kkr, delivery=0.000 kkr, export_tax=0.000 kkr, pretax=40.580 kkr, net_profit=28.406 kkr, demand=35
Panda    -> US      : cost(material+labor)=65.420 kkr, delivery=3.500 kkr, export_tax=3.750 kkr, pretax=77.330 kkr, net_profit=54.131 kkr, demand=40
Panda    -> Sweden  : cost(material+labor)=65.420 kkr, delivery=2.000 kkr, export_tax=0.000 kkr, pretax=44.580 kkr, net_profit=31.206 kkr, demand=2
500      -> Poland  : cost(material+labor)=84.694 kkr, delivery=0.000 kkr, export_tax=0.000 kkr, pretax=7.306 kkr, net_profit=6.210 kkr, demand=20
500      -> Italy   : cost(material+labor)=84.694 kkr, delivery=1.000 kkr, export_tax=0.000 kkr, pretax=50.306 kkr, net_profit=42.760 kkr, demand=40
500      -> US      : cost(material+labor)=84.694 kkr, delivery=2.800 kkr, export_tax=4.250 kkr, pretax=78.25

The objectibe function is

Z= sum c_ij*x_ij

In [12]:
import gurobipy as gp
from gurobipy import GRB
m2=gp.Model("Grobal market")
# task 1 requirement
models = ["Panda", "500", "Musa", "Giulia"]
min_prod_k = {m: min_prod[m] / 1000 for m in models}
pairs = list(data.keys())
# variables：x[i,j] = the number of the model sold to each country
x = m2.addVars(pairs, name="ship")
# objective function：maximize the net profit
m2.setObjective(gp.quicksum(c_market[p] * x[p] for p in pairs), GRB.MAXIMIZE)
# constraints
for p in pairs:
    m2.addConstr(x[p] >= demand[p], f"demand_{p[0]}_{p[1]}")
for i in models:
    m2.addConstr(
        gp.quicksum(x[i, j] for (m_name, j) in pairs if m_name == i) >= min_prod_k[i],
        name=f"min_prod_{i}"
    )
m2.addConstr(
    gp.quicksum(total_cost_market[p] * x[p] for p in pairs) <= 4e4,
    name="budget"
)

m2.optimize()
print("\n--- Optimal shipment plan ---")
for p in pairs:
    print(f"{p[0]:8s} -> {p[1]:8s}: {x[p].X:.2f} (thousand units)")

print(f"\nOptimal total net profit: {m2.objVal:.2f} kkr (thousand Kr)")



Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 19 rows, 14 columns and 42 nonzeros
Model fingerprint: 0x93cd4f9b
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [6e-01, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 4e+04]
Presolve removed 17 rows and 11 columns
Presolve time: 0.00s
Presolved: 2 rows, 3 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.4777447e+04   0.000000e+00   0.000000e+00      0s
       0    2.4777447e+04   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.477744713e+04

--- Optimal shipment plan ---
Panda    -> Poland  : 75.00 (thousand units)
Panda    -> Italy   : 35.00 (thousand units)
Panda    -> US      : 40.00 (thousand units)
Panda   

Task 3 Multi-line production

Since the foam is composed of three components, and the rate is 1:1:1. Therefore, the maximum quantities of foam produced is decided by the minumum component. If the decision variable is x_ij, i represents component 1,2,3 and j represents which line produces. Then, we introduce a variable F, the total produced quantities of foam, the linear objective function is MAX F, with three lower than constraints.

In [13]:
import gurobipy as gp
from gurobipy import GRB
line=[1,2,3,4]
component=[1,2,3]
capacity={1:100,2:150,3:80,4:200}
rate={
    (1,1):10,(1,2):15,(1,3):5,
    (2,1):15,(2,2):10,(2,3):5,
    (3,1):20,(3,2):5,(3,3):10,
    (4,1):10,(4,2):15,(4,3):20,
}
#(lines,component):rate
m3=gp.Model("Multi-line production")
x=m3.addVars(line,component,name="hours")
F=m3.addVar(name="foam")
m3.setObjective(F,GRB.MAXIMIZE)
for j in component:
    m3.addConstr(F<=gp.quicksum(rate[i,j]*x[i,j] for i in line),f"foam_limit_comp{j}")
for i in line:
    m3.addConstr(gp.quicksum(x[i,j] for j in component)<=capacity[i],f"capacity_line{i}")
m3.optimize()
print("\n--- Hours allocated ---")
for l in line:
    for j in component:
        print(f"Line {l}, Component {j}: {x[l,j].X:.2f} hours")

print(f"\nMax foam produced: {F.X:.2f} units")

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 7 rows, 13 columns and 27 nonzeros
Model fingerprint: 0xefff5a9a
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+01, 2e+02]
Presolve time: 0.00s
Presolved: 7 rows, 13 columns, 27 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.6000000e+31   3.000000e+30   1.600000e+01      0s
       9    2.9241379e+03   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.00 seconds (0.00 work units)
Optimal objective  2.924137931e+03

--- Hours allocated ---
Line 1, Component 1: 0.00 hours
Line 1, Component 2: 100.00 hours
Line 1, Component 3: 0.00 hours
Line 2, Component 1: 88.28 hours
Line 2, Component 2: 61.72 hours
Line 2, Component 3: 0.0

Task 5: Investments optimization

In [18]:

def solve_investments():
    # Monetary unit: one million crowns
    securities = ["A", "B", "C", "D", "E"]
    budget = 1000.0

    duration = {
        "A": 9, "B": 15, "C": 4, "D": 3, "E": 2
    }
    risk = {
        "A": 2, "B": 3, "C": 1, "D": 4, "E": 5
    }
    after_tax_return = {
        "A": 0.045,
        "B": 0.054,
        "C": 0.051 * (1 - 0.30),
        "D": 0.044 * (1 - 0.30),
        "E": 0.061,
    }

    # Explicit margins for strict requirements
    eps_duration = 1e-4
    eps_risk = 1e-4
    eps_A = 0.001  # 1000 crowns above the threshold

    model = gp.Model("investments")

    # Investment amounts and binary switches
    x = model.addVars(
        securities, lb=0.0, ub=budget,
        vtype=GRB.CONTINUOUS, name="investment"
    )
    y = model.addVars(
        ["C", "D", "E"], vtype=GRB.BINARY, name="enabled"
    )

    # Maximize after-tax return
    model.setObjective(
        gp.quicksum(
            after_tax_return[i] * x[i] for i in securities
        ),
        GRB.MAXIMIZE,
    )

    model.addConstr(
        gp.quicksum(x[i] for i in securities) == budget,
        name="full_investment",
    )

    model.addConstr(
        x["B"] + x["C"] + x["D"] >= 0.40 * budget,
        name="public_government_share",
    )

    model.addConstr(
        gp.quicksum(duration[i] * x[i] for i in securities)
        <= (5 - eps_duration) * budget,
        name="average_duration",
    )

    model.addConstr(
        gp.quicksum(risk[i] * x[i] for i in securities)
        <= (1.5 - eps_risk) * budget,
        name="average_risk",
    )

    # C and D are mutually exclusive
    model.addConstr(x["C"] <= budget * y["C"], name="link_C")
    model.addConstr(x["D"] <= budget * y["D"], name="link_D")
    model.addConstr(y["C"] + y["D"] <= 1, name="exclusive_CD")

    # Investment in E requires A above one million crowns
    model.addConstr(x["E"] <= budget * y["E"], name="link_E")
    model.addConstr(
        x["A"] >= (1 + eps_A) * y["E"],
        name="E_requires_A",
    )

    model.Params.MIPGap = 0.0
    model.optimize()

    if model.SolCount == 0:
        print(f"No feasible solution available. Status: {model.Status}")
        return model

    print("\nProven optimal:", model.Status == GRB.OPTIMAL)
    print("Investment amounts (million crowns):")

    for i in securities:
        print(f"  {i}: {x[i].X:.6f}")

    total = sum(x[i].X for i in securities)
    avg_duration = sum(
        duration[i] * x[i].X for i in securities
    ) / total
    avg_risk = sum(
        risk[i] * x[i].X for i in securities
    ) / total
    public_government = x["B"].X + x["C"].X + x["D"].X

    print(f"Total investment: {total:.6f}")
    print(f"After-tax return: {model.ObjVal:.6f} million crowns")
    print(f"Portfolio return rate: {100 * model.ObjVal / total:.6f}%")
    print(f"Average duration: {avg_duration:.6f} years")
    print(f"Average risk: {avg_risk:.6f}")
    print(
        "Public/government share: "
        f"{100 * public_government / budget:.6f}%"
    )

    return model


if __name__ == "__main__":
    solve_investments()

Set parameter MIPGap to value 0
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
MIPGap  0

Optimize a model with 9 rows, 8 columns and 28 nonzeros
Model fingerprint: 0xd1abf130
Variable types: 5 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [3e-02, 6e-02]
  Bounds range     [1e+00, 1e+03]
  RHS range        [1e+00, 5e+03]
Presolve removed 4 rows and 3 columns
Presolve time: 0.00s
Presolved: 5 rows, 5 columns, 14 nonzeros
Variable types: 4 continuous, 1 integer (1 binary)
Found heuristic solution: objective 37.5598140

Root relaxation: objective 3.953792e+01, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0   